## DATA 37711 Homework 3
### Glen Morgenstern

## Problem 1
### ai.
Poisson gives us matching support. Y takes in values {0, 1, 2, ...} and Poisson is supported on exactly those values. It's also the canoncial model for count data and has a single parameter so we only need to model the conditional mean.

### aii.
The Poisson rate $\mu$ must be strictly positive, but $f(X) = \sum_j \lambda_j h_j(X)$ is a sum of regression trees and can take any real value. The link function must therefore map $\mathbb{R} \to (0,\infty)$. The exponential does this, and it is the canonical link for the Poisson.

### 1b.
Under the model, $Y \mid X \sim \text{Poi}(\mu)$ with $\mu = \exp(f(X))$, so
$$
P(Y = y \mid X = x) = \frac{\mu^{y} e^{-\mu}}{y!} = \frac{\exp\!\big(y f(x)\big)\exp\!\big(-e^{f(x)}\big)}{y!}.
$$
The negative log likelihood is
$$
-\log P(Y \mid X) = e^{f(X)} - Y f(X) + \log(Y!).
$$
The term $\log(Y!)$ does not depend on $f$, so it can be dropped. The per-example loss is therefore
$$
L\big(f(x), Y\big) = e^{f(x)} - Y f(x),
$$
and we fit $f$ by minimizing the empirical risk
$$
\hat{R}(f) = \frac{1}{n}\sum_{i=1}^{n}\Big( e^{f(x_i)} - y_i f(x_i) \Big).
$$
This loss is convex in $f(x)$.

### 1c.
Differentiating per-example loss $L(f(x), Y) = e^{f(x)} - Y f(x)$ with respect to $f(x)$ gives
$$
\frac{dL}{df(x)} = e^{f(x)} - Y.
$$
This is the predicted mean $\hat{\mu} = e^{f(x)}$ minus the observed count $Y$. The negative gradient, $Y - e^{f(x)}$, is therefore the residual on the mean scale.

### 1d.
First order boosting is gradient descent in function sapce. We build f greedily, one tree at a time.

1. Initialize with the best constant, $f_0(x) = \log \bar{y}$, where $\bar{y} = \frac{1}{n}\sum_i y_i$ (this minimizes $\sum_i e^{c} - y_i c$ over constants $c$)
2. For $j = 1, \dots, J$:
  - Compute negative gradients at current model:
$$
r_i^{(j)} = -\frac{dL}{df(x_i)}\bigg|_{f = f_{j-1}} = y_i - e^{f_{j-1}(x_i)}.
$$
  - Fit a regression tree $h_j$ to $\{(x_i, r_i^{(j)})\}_{i=1}^n$ by ordinary least squares, i.e.
$$
h_j = \arg\min_{h} \sum_{i=1}^n \big( r_i^{(j)} - h(x_i) \big)^2 .
$$
  - Choose the step size $\lambda_j$, either as a fixed small learning rate or by line search
$$
\lambda_j = \arg\min_{\lambda} \sum_{i=1}^n L\big(f_{j-1}(x_i) + \lambda h_j(x_i),\, y_i\big).
$$
  - Update $f_j(x) = f_{j-1}(x) + \lambda_j h_j(x)$.

3. The final model is $f(x) = f_0(x) + \sum_{j=1}^J \lambda_j h_j(x)$, with predicted mean $\hat{\mu}(x) = e^{f(x)}$.

## Problem 2a.

Let $f_1,\dots,f_k$ be the models fit on the training set, one for each hyperparameter choice, and assume the loss satisfies $\ell \in [0,1]$. Let $R(f) = \mathbb{E}[\ell(f(X),Y)]$ be the true risk and $\hat R_{\text{test}}(f) = \frac{1}{l}\sum_{m=1}^{l}\ell(f(X_m),Y_m)$ the risk on the test set, which is drawn i.i.d. and independently of the training set. Let $\hat f = \arg\min_i \hat R_{\text{test}}(f_i)$ be the selected model.

Claim. With probability at least $1-\delta$ over the test set,
$$
\big|\hat R_{\text{test}}(\hat f) - R(\hat f)\big| \le \sqrt{\frac{\log(2k/\delta)}{2l}}.
$$

Proof. Condition on the training set, so that $f_1,\dots,f_k$ are fixed. For each fixed $i$, the test losses $\ell(f_i(X_m),Y_m)$ are i.i.d., lie in $[0,1]$, and have mean $R(f_i)$. Hoeffding's inequality gives, for any $\epsilon>0$,
$$
P\big(|\hat R_{\text{test}}(f_i) - R(f_i)| > \epsilon\big) \le 2e^{-2l\epsilon^2}.
$$
The selected model $\hat f$ depends on the test set, so Hoeffding cannot be applied to it directly. Instead we control all $k$ models at once with a union bound:
$$
P\big(\exists\, i:\ |\hat R_{\text{test}}(f_i) - R(f_i)| > \epsilon\big) \le 2k\,e^{-2l\epsilon^2}.
$$
Setting the right-hand side equal to $\delta$ and solving for $\epsilon$ gives $\epsilon = \sqrt{\log(2k/\delta)/(2l)}$. Hence with probability at least $1-\delta$, \emph{every} $f_i$, and in particular $\hat f$, satisfies the bound. Since this holds conditionally on every training set, it holds unconditionally. $\qquad\blacksquare$

## 2b

The model's test error will be optimistically biased. If we choose our model based on minimizing test risk, then we will see our answer is partly luck. The model whose noise happened to fall favorably on this particular test will get chosen. The bias will grow with k relative to l. Each $\hat R_{test}(f_i)$ fluctuates around $R(f_i)$ at scale about $1/\sqrt(l)$. When k is large relative to l, the expected size of favoarable noise is not small, and the minimum can be driven down by noise alone. The test score will say little about the model's real performance.

## 2c

In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

SEED = 0
df = pd.read_csv("winequality-white.csv", sep=";")
X, y = df.drop(columns="quality").values, df["quality"].values

# 70% train, 10% validation, 20% test
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size = 0.30, random_state = SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size = 2/3, random_state = SEED)

def grid_search(make_model, grid):
    results = []
    for params in grid:
        m = make_model(params).fit(X_train, y_train)
        results.append((mean_squared_error(y_val, m.predict(X_val)), params))
    return sorted(results, key=lambda t: t[0])

knn_grid = ParameterGrid({"n_neighbors": list(range(1, 71)),
                          "weights": ["uniform", "distance"], "p": [1, 2, 3, 4]}) # 560
rf_grid = ParameterGrid({"n_estimators": [10, 20, 30 , 40, 50],
                         "max_depth": [None, 4, 8, 12, 16, 20],
                         "min_samples_leaf": [1, 2, 4, 8],
                         "max_features": [0.2, 0.4, 0.6, 0.8, 1.0]}) # 600
make_knn = lambda p: make_pipeline(StandardScaler(), KNeighborsRegressor(**p))
make_rf = lambda p: RandomForestRegressor(random_state = SEED, n_jobs = -1, **p)

for name, make, grid in [("KNN", make_knn, knn_grid), ("RF", make_rf, rf_grid)]:
    res = grid_search(make, grid)
    best_val, best_p = res[0]
    best = make(best_p).fit(X_train, y_train)
    test_mse = mean_squared_error(y_test, best.predict(X_test))
    print(f"{name}: {best_p}\n val MSE {best_val:.4f}, test MSE {test_mse:.4f}")
    

KNN: {'n_neighbors': 16, 'p': 1, 'weights': 'distance'}
 val MSE 0.4000, test MSE 0.4615
RF: {'max_depth': None, 'max_features': 0.6, 'min_samples_leaf': 1, 'n_estimators': 50}
 val MSE 0.3825, test MSE 0.4609


For both models, the test MSE is higher than the validation MSE of the selected configuration. This is selection bias. The winner was picked as the minmum over 560 or 600 noisy validation estimates, so its validation score is optimistic, $\mathbb{E}[\min_i \hat R_{\text{val}}(f_i)] \le \mathbb{E}[R(\hat f)]$. The test set played no role in selection, so its estimate is unbiased. Across the grid the validation MSE ranges from 0.40 to 0.71 for KNN and 0.38 to 0.58 for RF.

## Problem 3

In [4]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss

for name in ["dset0", "dset1"]:
    d = pd.read_csv(f"{name}.csv", header=None)
    X, y = d.iloc[:, 1:].values, d.iloc[:, 0].values
    res = {n: [] for n in [1, 10, 100, 300, 1000]}
    for seed in range(5):
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size = 0.3, random_state = seed)
        for n in res:
            m = GradientBoostingClassifier(n_estimators = n, random_state = 0).fit(Xtr, ytr)
            res[n].append((accuracy_score(ytr,m.predict(Xtr)), accuracy_score(yte,m.predict(Xte)), 
                           log_loss(ytr,m.predict_proba(Xtr)), log_loss(yte,m.predict_proba(Xte))))
    print(name, "| trees | train acc | test acc | train logloss | test logloss")
    for n, r in res.items():
        print(n, np.round(np.mean(r, 0), 3))

dset0 | trees | train acc | test acc | train logloss | test logloss
1 [0.574 0.575 0.652 0.666]
10 [0.849 0.736 0.492 0.58 ]
100 [0.997 0.825 0.128 0.397]
300 [1.    0.839 0.021 0.45 ]
1000 [1.    0.829 0.    0.892]
dset1 | trees | train acc | test acc | train logloss | test logloss
1 [0.702 0.693 0.639 0.646]
10 [0.883 0.797 0.405 0.471]
100 [0.99  0.817 0.108 0.388]
300 [1.    0.815 0.019 0.512]
1000 [1.    0.8   0.    1.021]


## 3a
Training accuracy reaches about 100% by 100 trees. Test accuracy keeps improving up to 100 to 300 trees (0.83 to 0.84) and then drops slightly at about 1000. So test accuracy barely degrades even after the model interpolates the training data. However, test log-loss is best near 100 trees and then rises sharply from 0.40 to 0.89 because the model becomes overconfident.

## 3b
The pattern is similar to dset0 but less forgiving. Accuracy peaks around 100 trees (0.82), then declines to about 0.80 at 1000 trees, and test log-loss rises from 0.39 to 1.02.

## 3c
The difference in accuracy is small between the two. They both see test accuracy fall beyond some point. The only difference is that test accuracy begins to fall earlier (fewer trees) for dset1 compared to dset0. This could potentially mean that the true signal for dset0 is more complex, which means more trees can model it better than simpler (fewer trees) models. This means that dset1 could have a potentially less complex true data generating model, which explains why more complexity (more trees) starts to hurt earlier, as overfitting takes over faster.